# Intro to programming and training Neural Networks with PyTorch


<center><img src="figures/pytorch_logo.png"></center>

## Introduction

* PyTorch is an end-to-end open source platform for ML 
* Allows to easily build and deploy ML powered applications.
* Not only Neural Networks




## PyTorch

* PyTorch is an "optimized tensor library for deep learning"
* Scientific computing, general ML, Neural Networks
* C++/python (we use the latter)
* Easy to implement complex architectures with few lines of code

## Docs: https://docs.pytorch.org/docs

* Installation inscructions (you should be already set up!)
* Tutorials from the groud up
* Reference API
  * Models and Layers
  * Data handling
  * Helper functionalities (utils)


## Other ML/DL libraries

* Tensorflow/Keras
* JAX
* ...
* Most concepts translate across libraries with minor differences

## Some terminology

* A dataset in supervised learning is made of a number of (features, label) pairs
* Example, a dataset of diabetic patients is made of:
    * Features: information describing each patient (weight, height, blood pressure...)
    * Labels: whether each patient is diabetic or not (glucose levels higher or lower than...)
* Each (features, label) pair is also called a _sample_ or _example_. Basically a data point
* Features are also sometimes called _inputs_ when referred to something you feed to a NN
* Labels are compared to the NN's _outputs_ to see how well the network is doing compared to the truth

## What is a tensor

The main variables in PyTorch are tensors:

> A tensor is often thought of as a generalized matrix. That is, it could be a 1-D matrix (a vector), a 3-D matrix (something like a cube of numbers), even a 0-D matrix (a single number), or a higher dimensional structure that is harder to visualize. The dimension of the tensor is called its rank.
>
> Src: https://www.kaggle.com/discussions/getting-started/159424



## What is a tensor

The main variables in TensorFlow are, of course, tensors:

> A tensor is often thought of as a generalized matrix. That is, it could be a 1-D matrix (a vector), a 3-D matrix (something like a cube of numbers), even a 0-D matrix (a single number), or a higher dimensional structure that is harder to visualize. The dimension of the tensor is called its rank.

## PyTorch operates on tensors

> Tensors are a specialized data structure that are very similar to arrays and matrices. In PyTorch, we use tensors to encode the inputs and outputs of a model, as well as the model’s parameters.
> 
> Tensors are similar to NumPy’s ndarrays, except that tensors can run on GPUs or other hardware accelerators.
>
> Src: https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html

## The first step is to build a graph of operations

* NNs are defined in PyTorch as graphs through which the data flows until the final result is produced
* Before we can do any operation on our data (images, etc) we need to build the graph of tensor operations
* When we have a full graph built from input to output, we can run our data (training or testing) through it.

> (PyTorch implements) Over 1200 tensor operations, including arithmetic, linear algebra, matrix manipulation (transposing, indexing, slicing), sampling and more (...)


## Tensors and data are *not* the same thing
* Tensors are, rather, a symbolic representation of the data
* Think about the function $g = f(x)$: as long as we do not assign a value to $x$, we will not have a fully computed $g$
* In this case, $g$ is the output tensor, $x$ the input tensor, $f$ the tensor operation (a Neural Network?)


## Example

* We have a set of color images of size $1000x1000$ pixels (1 megapixel) that we want to use on our NN 
* We define tensors with shape $(n, 1000, 1000, 3)$
    * $n$ is the number of images that we are presenting to our network in one go (a "batch")
    * $1000x1000$: image pixels
    * $3$ is the number of channels (RGB)
    * Grayscale images tensors would have shape $(n, 1000, 1000, 1)$

## One thing to remember when operating on tensors

The dimensions between tensors coming out of the $i$-th node and those going into the $(i+1)$-th node *must* match:

* If each sample in our dataset is made of 10 features, the first (input) layer must accept a tensor of shape $(n, 10)$
* If the first layer in our NN outputs a 3D tensor, the second layer must accept a 3D tensor as input
* Check the documentation to make sure what input-output shapes are allowed ([example](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv1d.html))

## Here's how a NN layer might look like in PyTorch:

* 7 samples in batch
* 784 inputs
* 500 outputs

<center><img src="figures/nn_layer_tboard.png"></center>


## Here is how a model is built and trained


```python
import torch
import torch.nn as nn


#Multi-layer perceptron (one hidden layer)
model = nn.Sequential()
model.append(nn.Linear(3, 3))
model.append(nn.Sigmoid())
model.append(nn.Linear(3, 1))
model.append(nn.Sigmoid())

#Gradient descent algorithm, Mean Squared Error as Loss function
loss = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

dataset = ... # define dataset. More on this later

for features, label in dataset:
    optimizer.zero_grad()
    prediction = model(features)
    loss = loss_fn(prediction, label)
    loss.backward()
    optimizer.step()
...
```

What does each bit do?

## A neural network in PyTorch is called a Model

The simplest kind of model is of the Sequential kind:

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

model = nn.Sequential()
print(list(model.parameters()))

[]


This is an "empty" model, with no layers, no inputs or outputs are defined either.

Adding layer is easy. Let's say we have data for participants to a clinical study. For participant we have recorded: blood pressure, BMI and age.

The participants have been diagnosed as healthy or sick, these will be our labels.

We could define a simple NN that predicts if a participant is healthy or sick as follows:

In [2]:
model = nn.Sequential()
model.append(nn.Linear(3, 4))
model.append(nn.ReLU())
model.append(nn.Linear(4, 2))
model.append(nn.Softmax())

Sequential(
  (0): Linear(in_features=3, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=2, bias=True)
  (3): Softmax(dim=None)
)


A "Dense" layer is a fully connected layer as the ones we have seen in Multi-layer Perceptrons.
The above is equal to having this network:

<center><img src="figures/simplenet_patient.png"></center>


If we want to see the layers in the Model this far, we can just call:

In [3]:
list(model.parameters())

[Parameter containing:
 tensor([[-0.1700,  0.2913,  0.1137],
         [-0.3291,  0.3371,  0.0717],
         [ 0.1786, -0.2174, -0.3721],
         [-0.1104,  0.1126,  0.1228]], requires_grad=True),
 Parameter containing:
 tensor([-0.1642,  0.2731,  0.0200,  0.4404], requires_grad=True),
 Parameter containing:
 tensor([[ 0.4967, -0.0760,  0.2080, -0.0456],
         [ 0.3384,  0.1176,  0.1223, -0.0560]], requires_grad=True),
 Parameter containing:
 tensor([ 0.3756, -0.2102], requires_grad=True)]

Notice the number of parameters, can you tell why there are this many?

Using "model.append()" keeps stacking layers on top of what we have:

In [4]:
model.append(nn.Linear(2, 2))
model

Sequential(
  (0): Linear(in_features=3, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=2, bias=True)
  (3): Softmax(dim=None)
  (4): Linear(in_features=2, out_features=2, bias=True)
)

One can also declare the model in one go, by passing a list of layers to Sequential() like so:

In [5]:
model = nn.Sequential(
    nn.Linear(3, 4),
    nn.ReLU(),
    nn.Linear(4, 2),
    nn.Sigmoid()
)

model

Sequential(
  (0): Linear(in_features=3, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=2, bias=True)
  (3): Sigmoid()
)

## Small exercise

* Can you write code to build a simple NN model?
* Open the `exercises` jupyter notebook

## PyTorch layers (https://docs.pytorch.org/docs/stable/nn.html)

Common layers (we will cover most of these!)

* Trainable
    * <font color='red'>Linear (fully connected/MLP)</font>
    * <font color='red'>Conv1D (2D/3D)</font>
    * <font color='red'>Recurrent: LSTM/GRU/Bidirectional</font>
    * <font color='red'>Embedding</font>
    * <font color='red'>Lambda (apply your own function)</font>

* Non-trainable
    * <font color='red'>Dropout</font>
    * <font color='red'>Flatten</font>
    * BatchNormalization
    * MaxPooling1D (2D/3D)
    * Merge (add/subtract/concatenate)
    * <font color='red'>Activation (Softmax/ReLU/Sigmoid/...)</font>


## More things needed to train the model

Once we have defined a model we need to at least define:

* a Loss function (calculate the prediction error against a set of labels)
* an Optimizer (the algorithm that finds the minimum loss possible).

In [6]:
loss = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

## Losses (https://docs.pytorch.org/docs/stable/nn.html#loss-functions)

These are the functions used to evaluate and train the neural network. Different losses are suited for different problems.

Loss for classification problems:
* Categorical Crossentropy

Loss to compare distributions:
* KL Divergence

Common losses for regression problems:
* Mean Squared Error
* Mean Absolute Error

A loss function is also called a `criterion` in pytorch code

## Metrics (https://lightning.ai/docs/torchmetrics/stable/)

Common metrics for classification:
* Accuracy
* Precision/Recall
* AUC

Common metrics for regression:
* Mean Squared Error
* Mean Absolute Error

## Metrics (https://lightning.ai/docs/torchmetrics/stable/)

* While a Loss function can tell us how the training is going, these measures are not always intuitive
* We sometimes what to have measures such as:
    * accuracy (how often do we get the classification right?)
    * AUC
    * Correlation coefficients
    * ...

In [2]:
from torch import tensor
from torchmetrics.classification import BinaryAccuracy

target = tensor([0, 1, 0, 1, 0, 1])
preds  = tensor([0, 0, 1, 1, 0, 1])

metric = BinaryAccuracy()
metric(preds, target)

tensor(0.6667)

## Optimizers (https://docs.pytorch.org/docs/stable/optim.html)

* They are algorithms for gradient descent
* A few to choose from:
    * SGD (Stochastic Gradient Descent)
    * RMSprop (Root Mean Square propagation)
    * Adadelta (Adaptive delta)
    * Adam (Adaptive Moment estimation)


<br>
<br>
<br>
<br>
<img src="figures/gradient_descent.png">

## Gradient Descent 

We have seen how gradient descent works:

For each epoch:
* Get predicted $y$ ($ŷ$) for all $N$ samples
* Calculate error (loss)
* Calculate all gradients (backprop)
* Apply gradients to weights
    
Pros/cons:
* Stable procedure
* Guarantees lower error at next step
* Will get stuck at local minimum

<br>
<br>
<br>
<br>
<img src="figures/gradient_descent.png">

## Stochastic Gradient Descent
For each epoch:
* Divide data in batch blocks of size $n < N$
* For each of the $N/n$ blocks:
    * Get predicted $y$ for $n$ samples
    * Calculate partial loss
    * Calculate gradients (backprop)
    * Apply gradients to weights

Pros/cons:
* Noisy gradients
* Error will still go down overall
* Less likely to get stuck at local minimum

<br>
<br>
<br>
<br>
<img src="figures/gradient_descent.png">

## Optimizers

We need to choose a learning rate to multiply to our gradient. If it is too small, we risk taking too long to get to a minimum
<center><img src="figures/small_lr.png"></center>

## Optimizers

If it is too large, the network risks becoming unstable, explode

<center><img src="figures/large_lr.png"></center>

Let's test different optimization strategies on Tensorflow playground: http://playground.tensorflow.org

## Optimizers

Luckily there are algorithms to address these issues:
* Increase descent speed when past gradients agree with current, slow down otherwise (momentum)
* Annealing (decrease learning rate with passing time)
* Different learning rates for different parameters
* Adaptive learning rate based on gradient

<br>
<br>
<br>
<br>
<img src="figures/adaptive_lr.png">

## Optimizers

* They are algorithms for gradient descent
* A few to choose from:
    * SGD (stochastic gradient descent)
        * One learning rate, fixed
        * Old, but works well with Nesterov momentum
    * RMSprop
        * One learning rate per parameter
        * Adaptive learning rate (divide by squared mean of past gradients)
    * Adadelta (adaptive learning rate)
        * Similar to RMSprop, no need to set initial learning rate
    * Adam (Adaptive moment estimation)
        * Combines pros from RMSprop, Adadelta, works well with most problems


## Optimizers
<br>
<br>
<center><img src="figures/adam_et_al.png" width=500></center>
<div style="text-align: right">("Adam: A Method for Stochastic Optimization", 2015)</div>

## Training the model

* We are almost ready to train the model, I swear
* Training is done in a loop over your data
* For each sample (features, label):
   * Predict: `prediction = model(features)`
   * Assess error: `loss = criterion(prediction, label)`
   * Backpropagate error, calculate gradients: `loss.backward()`
   * Take a step along the gradient direction: `optimizer.step()`

```python
for batch in dataset:
    features, labels = batch
    optimizer.zero_grad()
    prediction = model(features)
    loss = loss_fn(prediction, label)
    loss.backward()
    optimizer.step()
```

## Training the model, more in detail

In reality, there are a few more things to keep track of. Here is a more complete version of a training loop:

```python
# device: are we training on CPU or GPU?
device = None
if device is None:
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# send the model to whatever device we're using
model.to(device)
    
for epoch in range(max_epochs):
    # reset metrics
    training_loss_acc = 0
    training_examples = 0
    # put model in "training mode"
    model.train()

    # training loop, one epoch
    for i, batch in enumerate(train_loader):
        optimizer.zero_grad()
        
        x_batch, y_batch = batch
        # send data to whatever device we're using
        x_batch = x_batch.to(device)  
        y_hat = model(x_batch)

        loss = criterion(y_hat, y_batch.to(device))
        loss.backward()

        optimizer.step()
        training_loss_acc += loss.item()
        training_examples += x_batch.size(0)
```

## Training the model, more in detail

* Let's not forget the actual data!
* First, we define a `Dataset` as a set of tensor features and labels
* Then, we define a `DataLoader` process the `Dataset` before handing it to the network
* We can also split our data into two or more parts that will be used for different purposes

```python
train_data = Dataset(...)
validation_data = Dataset(...)

train_dataloader = DataLoader(train_data, ...)
val_dataloader = DataLoader(validation_data, ...)

for epoch in range(max_epochs):
    ...
    for features, label in train_dataloader:
        ... # train your model

    model.eval() # put the model in validation mode
    with torch.no_grad(): # avoids computing stuff not needed at validation time
        for val_batch in val_dataloader:
            ... # evaluate your model
```

## What is this validation thing? Do I really need it?

* Yes, yes you do
* Helps understanding if the model is learning anything useful
* Take some of your labelled data, set it aside, call it **validation set** and don't train on it
* Also called a **development set**
* Evaluate model on validation set at the end of each epoch, see if model works on unseen data
* If it works well on training set but not on validation set, you're overfitting

<img src="figures/overfitting_class.png" width=300>

## What is this validation thing? Do I really need it?

* If it works well on training set but not on validation set, you're overfitting
* Validation (or development) data is used to adapt hyperparameters, select best models
* Validation (or development) data is **NOT** testing data (more on this later)
* Let's try this on Tensorflow playground: http://playground.tensorflow.org

<img src="figures/early_stopping.png" width=500>

## Ok, can we PLEASE train a NN now?

* Let's generate some artificial data, see what happens
* Classification dataset, 2 classes
* Let's say 10,000 samples, three features per sample
* Random data

In [3]:
import numpy as np

# Generate dummy data
data = np.random.random((10000, 3))
labels = np.random.randint(2, size=(10000))

#let's print the first sample (three floats) and its corresponding label:
print(np.hstack((data[0:10,:], labels[..., None][0:10])))

[[0.18950424 0.74664372 0.57298487 1.        ]
 [0.17924201 0.63454853 0.09023677 1.        ]
 [0.81525709 0.37936956 0.57452399 1.        ]
 [0.40678884 0.68700531 0.84553481 1.        ]
 [0.31415166 0.03544495 0.24370068 0.        ]
 [0.03301763 0.59545616 0.27065564 0.        ]
 [0.35112056 0.90216875 0.41183708 1.        ]
 [0.4728558  0.44368994 0.13419016 0.        ]
 [0.50198966 0.71255315 0.88821093 0.        ]
 [0.89946873 0.35925413 0.48854941 0.        ]]


## We have the data. Now we make the model, train it

* Batch size is 32, 10 epochs
* Take 10% of the data, reserve it for validation

In [4]:
from torch.utils.data import TensorDataset, DataLoader
import torchmetrics

device = None
if device is None:
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

max_epochs = 10
# convert the data and labels to tensors
tdata = torch.Tensor(data)
tlabels = torch.Tensor(labels).long()

model = nn.Sequential(
    nn.Linear(3, 3),
    nn.Sigmoid(),
    nn.Linear(3, 2),
)

model.to(device)

criterion = nn.CrossEntropyLoss()
metric = BinaryAccuracy()
optimizer = optim.Adam(model.parameters())

dataset = TensorDataset(tdata, tlabels)
# split the data randomly
train_set, dev_set = torch.utils.data.random_split(dataset, [9000, 1000])

# shuffle data at training time
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
dev_loader = DataLoader(dev_set, batch_size=32)
metric = torchmetrics.Accuracy(task='multiclass', num_classes=2, top_k=1)

for epoch in range(max_epochs):
    training_loss_acc = 0
    training_examples = 0
    model.train()
    for i, batch in enumerate(train_loader):
        x_batch, y_batch = batch
        x_batch = x_batch.to(device)
        y_hat = model(x_batch)

        loss = criterion(y_hat, y_batch.to(device))
        loss.backward()
        optimizer.step()
        training_loss_acc += loss.item()
        training_examples += x_batch.size(0)

    # training done for this epoch, validate:
    model.eval()
    with torch.no_grad():
        dev_loss = 0
        dev_acc = 0
        dev_examples = 0
        for i, batch in enumerate(dev_loader):
            x_batch, y_batch = batch
            x_batch = x_batch.to(device)
            y_hat = model(x_batch)
            dev_loss += criterion(y_hat, y_batch.to(device)).item()
            dev_examples += x_batch.size(0)
            dev_acc += metric(torch.argmax(y_hat, -1), y_batch)
        print(f"Train loss: {training_loss_acc/training_examples}, dev loss: {dev_loss/dev_examples}, dev acc: {dev_acc.item() / (i+1):.2f}")

        

Train loss: 0.02195857560634613, dev loss: 0.022169213593006134, dev acc: 0.51
Train loss: 0.022153997076882256, dev loss: 0.022399335265159605, dev acc: 0.49
Train loss: 0.022006620142194958, dev loss: 0.0221649928689003, dev acc: 0.50
Train loss: 0.02191546430852678, dev loss: 0.02221563923358917, dev acc: 0.49
Train loss: 0.02198013440767924, dev loss: 0.022227314054965974, dev acc: 0.51
Train loss: 0.021961797151300644, dev loss: 0.02278799831867218, dev acc: 0.49
Train loss: 0.022017470121383668, dev loss: 0.022665292978286743, dev acc: 0.51
Train loss: 0.021996690862708623, dev loss: 0.022827565968036652, dev acc: 0.49
Train loss: 0.022108186430401272, dev loss: 0.022197521686553954, dev acc: 0.49
Train loss: 0.02221493074629042, dev loss: 0.0224978044629097, dev acc: 0.51


## Let's visualize our training curves

* Plots loss and accuracy for train and validation sets separately


In [5]:
from typing import Optional
import plotly.graph_objects as go
 

class LivePlot():
    def __init__(self):
        self.fig = fig = go.FigureWidget()
        self.plot_indices = {}
        display(self.fig)
        self.limits = [0,0]
        self.current_x = 0
        
    def report(self, name: str, value: float):
        "Report new value for line `name` of the current time step. Use "
        try:
            plot_index = self.plot_indices[name]
        except KeyError:
            plot_index = len(self.fig.data)
            self.fig.add_scatter(y=[], x=[], name=name)
            self.plot_indices[name] = plot_index
        self.fig.data[plot_index].y += (value,)
        self.fig.data[plot_index].x += (self.current_x,)

    def increment(self, n_ticks: int):
        "Increment the currently displayed limits with these many ticks"
        self.limits[1] += n_ticks
        self.fig.update_layout(xaxis_range=self.limits)
    
    def set_limit(self, n_ticks: int):
        "Update the currently displayed to exactly these many ticsk"
        self.limits[1] = n_ticks
        self.fig.update_layout(xaxis_range=self.limits)

    def tick(self, n_ticks: Optional[int] = None):
        "Update the current time with these many ticks, or 1 tick if no argument is supplied."
        if n_ticks is None:
            n_ticks = 1
        self.current_x += n_ticks


# Cleaning up the code a bit

We move the training loop into its own function:

In [6]:
def train(*,
          model: torch.nn.Module, 
          train_loader: DataLoader, 
          dev_loader: DataLoader, 
          optimizer: torch.optim.Optimizer, 
          criterion: torch.nn.Module, 
          max_epochs: int,
          device: Optional[torch.device] = None,  
          liveplot: Optional[LivePlot]=None):
    if device is None:
        device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

    model.to(device)
    for epoch in range(max_epochs):
        training_loss_acc = 0
        training_examples = 0
        model.train()
        
        for i, batch in enumerate(train_loader):
            optimizer.zero_grad()
            
            x_batch, y_batch = batch
            x_batch = x_batch.to(device)  
            y_hat = model(x_batch)

            loss = criterion(y_hat, y_batch.to(device))
            loss.backward()

            optimizer.step()
            training_loss_acc += loss.item()
            training_examples += x_batch.size(0)
        
        model.eval()
        with torch.no_grad():
            dev_loss_acc = 0
            dev_examples = 0
            for batch in dev_loader:
                x_batch, y_batch = batch
                x_batch = x_batch.to(device)
                y_hat = model(x_batch)
                dev_loss_acc += criterion(y_hat, y_batch.to(device)).item()
                dev_examples += x_batch.size(0)
        
        if liveplot is not None:
            liveplot.tick() # Update the liveplot time
            liveplot.report("Training loss", training_loss_acc / training_examples)
            liveplot.report("Development loss", dev_loss_acc / dev_examples)

Then, we make the model into a `torch.nn.Module` class:

In [7]:
class MLP(torch.nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Sigmoid(),
            nn.Linear(hidden_dim, output_dim),
        )
    def forward(self, x):
        y_hat = self.layers(x)
        return y_hat

## Let's visualize our training curves

* Plots loss and accuracy for train and validation sets separately
* The model didn't learn anything, which makes sense (data is random)

In [11]:
# Setup plot
liveplot = LivePlot()

# model from MLP class
model = MLP(input_dim=3, output_dim=2, hidden_dim=3)

learning_rate = 1e-3
weight_decay = 1e-5
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
criterion = nn.CrossEntropyLoss()
epochs = 20
liveplot.increment(20)
train(model=model, train_loader=train_loader, dev_loader=dev_loader, optimizer=optimizer, criterion=criterion, max_epochs=epochs, liveplot=liveplot)

FigureWidget({
    'data': [], 'layout': {'template': '...'}
})

## Do it again, but with data that actually means something

* A XOR function is not linear
* A perceptron is not able to separate XOR classes
* A MLP should be able to

<img src="figures/xor_table.jpg">


Let's generate data that is not just binary, but behaves like it:

* A positive (+) input behaves like a 1
* A negative (-) input behaves like a 0
* -0.5 $\oplus$ 0.2 $\oplus$ -0.1 => 1

In [13]:
# Generate XOR data
data = np.random.random((1000000, 3)) - 0.5
labels = np.zeros((1000000, 1))

labels[np.where(np.logical_xor(np.logical_xor(data[:,0] > 0, data[:,1] > 0), data[:,2] > 0))] = 1

#let's print some data and the corresponding label to check that they match the table above
for x in range(3):
    print("{0: .2f} xor {1: .2f} xor {2: .2f} equals {3:}".format(data[x,0], data[x,1], data[x,2], labels[x,0]))

 0.01 xor  0.47 xor  0.21 equals 1.0
-0.36 xor -0.28 xor  0.47 equals 1.0
 0.43 xor  0.05 xor  0.00 equals 1.0


In [14]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
transformed = pca.fit_transform(data)
plt.scatter(transformed[:,0], transformed[:,1], c=labels)

ModuleNotFoundError: No module named 'sklearn'

Now let's fit a model to the data:

In [ ]:
from keras.layers import LeakyReLU, Dropout
model = Sequential()
model.add(Dense(16, input_dim=3, activation="tanh"))
model.add(Dense(8, activation="tanh"))
model.add(Dense(4, activation="tanh"))
model.add(Dense(2, activation='softmax'))

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# Train the model, iterating on the data in batches of 32 samples
history = model.fit(data, labels, epochs=2, batch_size=128, validation_split=0.1, shuffle=True)

In [ ]:
from keras.layers import LeakyReLU, Dropout
model = Sequential()
model.add(Dense(16, input_dim=3, activation="tanh"))
model.add(Dense(8, activation="tanh"))
model.add(Dense(4, activation="tanh"))
model.add(Dense(2, activation='softmax'))

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# Train the model, iterating on the data in batches of 32 samples
history = model.fit(data, labels, epochs=100, batch_size=128, validation_split=0.1, shuffle=True)

## XOR data

* Better than random!
* Notice the difference between train and validation curves

In [ ]:
plot_loss_acc(history)

## Exercise: can you do better?

* Check the exercise notebook!

In [ ]:
model = Sequential()
model.add(Dense(4, input_dim=3, activation='sigmoid'))
model.add(Dense(3, activation='sigmoid'))
model.add(Dense(2, activation='softmax'))

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# Train the model, iterating on the data in batches of 32 samples
history = model.fit(data, labels, epochs=10, batch_size=32, validation_split=0.1)